# Regresión: Predicción de precios Audi A3

**Rol de canalización:** Cuaderno obligatorio 04, el paso de modelado expected-price.

Este cuaderno entrena modelos de regresión simples y escribe resultados expected-price en BigQuery. Es un cuaderno básico: explícito, mínimo y fácil de revisar para los estudiantes después de clase.

**Consume:** registros de modelado de BigQuery `vw_regression_dataset`.
**Produce:** diagramas de dispersión guardados y `fact_expected_price_predictions`.
**Feeds:** `vw_bi_dashboard` y la capa de soporte de decisiones Streamlit.

Estimaciones de regresión del precio de mercado esperado. La brecha expected-price es una señal de revisión, no una decisión de adquisición automática.

### Descripción general funcional
Entradas: registros de modelado de BigQuery `vw_regression_dataset`.
Proceso: filtrar según el alcance configurado, visualizar características, entrenar modelos, comparar métricas, publicar precios esperados.
Salidas: diagramas de dispersión guardados en el disco y una tabla de predicción BigQuery.
Motivo: la regresión lineal proporciona una línea base expected-price interpretable.

**Objetivo:** Construir modelos de regresión lineal para estimar el precio esperado.
**Entradas:** Datos de listado con alcance de BigQuery.
**Proceso:** Aplicar el alcance configurado, entrenar modelos y conservar los resultados.
**Salidas:** Parcelas y `fact_expected_price_predictions`.
**Por qué:** Los resultados de precio esperado son consumidos por el soporte de decisiones BI.


## 1. Importaciones y configuración

Toda la configuración está agrupada en la parte superior para reducir la carga cognitiva.

### Configuración
Importamos las herramientas de modelado, fijamos una semilla aleatoria para la reproducibilidad y configuramos el proyecto/conjunto de datos BigQuery. Las carpetas de salida se crean para que los gráficos y archivos guardados tengan ubicaciones estables.

### Opciones de modelo
Usamos regresión lineal simple para la interpretabilidad. Los informes MAE y R2 mantienen la evaluación simple y alineada con los objetivos de aprendizaje para principiantes.

**Objetivo:** Establecer importaciones, semillas aleatorias y rutas de salida.
**Entradas:** Bibliotecas de modelado e identificadores BigQuery.
**Proceso:** Importar módulos y crear directorios necesarios.
**Resultados:** Entorno configurado para modelado y generación de informes.
**Por qué:** La configuración estable reduce los errores y mejora la reproducibilidad.


In [ ]:
# Objetivo: importar bibliotecas, cargar configuraciones compartidas y preparar carpetas de salida.  #Objetivo
# Entrada: bibliotecas de modelado, cliente BigQuery y valores project_config.yaml.  #Aporte
# Proceso: importar módulos, analizar la configuración, establecer el alcance y la semilla aleatoria, crear un directorio de salida.  #Proceso
# Salida: Entorno inicializado para análisis de regresión en un ámbito configurado.  #Producción

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

from google.cloud import bigquery


def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / '.git').exists() or (p / 'config' / 'project_config.yaml').exists():
            return p
    return start


def load_project_config(config_path: Path) -> dict:
    config = {}
    if not config_path.exists():
        return config
    for raw_line in config_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or ':' not in line:
            continue
        key, value = line.split(':', 1)
        key = key.strip()
        value = value.strip()
        if value.startswith(('"', "'")) and value.endswith(('"', "'")):
            value = value[1:-1]
        elif value.lower() in ('true', 'false'):
            value = value.lower() == 'true'
        else:
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    pass
        config[key] = value
    return config


PROJECT_ROOT = find_repo_root(Path.cwd())
PROJECT_CONFIG = load_project_config(PROJECT_ROOT / 'config' / 'project_config.yaml')

RANDOM_SEED = int(PROJECT_CONFIG.get('random_state', 42))
np.random.seed(RANDOM_SEED)

project_id = str(PROJECT_CONFIG.get('gcp_project_id', 'albertheadofdata101')).strip()
dataset_id = str(PROJECT_CONFIG.get('bq_dataset', 'autoscout_audi_a3_germany')).strip()
make = str(PROJECT_CONFIG.get('make', 'audi')).strip().lower()
model = str(PROJECT_CONFIG.get('model', 'a3')).strip().lower()
country_name = str(PROJECT_CONFIG.get('country', 'germany')).strip().lower()
TAG = f'{make}_{model}_{country_name}'

REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


## 2. Cargar datos de modelado desde BigQuery

Extraemos datos directamente de BigQuery para que los estudiantes utilicen la misma fuente de verdad.

### Carga de datos
Consultamos `vw_regression_dataset` y aplicamos el alcance de marca/modelo/país configurado de `project_config.yaml`.

### Por qué es importante el alcance explícito
El uso de un alcance explícito mantiene controlada la cuestión empresarial y evita cambios de alcance automáticos ocultos.

**Objetivo:** Cargar el conjunto de datos de modelado de BigQuery.
**Entradas:** `vw_regression_dataset` y valores de alcance configurados.
**Proceso:** Consulta la vista con filtros de alcance y valida la disponibilidad de filas.
**Salidas:** `df` con funciones de modelado y objetivo.
**Por qué:** El control del alcance es parte de la prestación analítica profesional.


In [ ]:
# Objetivo: cargar datos de modelado de BigQuery para el alcance configurado.  #Objetivo
# Entrada: project_id, dataset_id y valores configurados de marca/modelo/país.  #Aporte
# Proceso: consulte vw_regression_dataset con filtros de alcance explícitos y valide el recuento de filas.  #Proceso
# Salida: df que contiene el conjunto de datos de modelado utilizado para la regresión del precio esperado.  #Producción

client = bigquery.Client(project=project_id)


def run_query(sql):
    return client.query(sql).to_dataframe()


sql_modeling = f'''
SELECT
  listing_id,
  actual_price_eur AS price_eur,
  mileage_km,
  age_years,
  power_hp,
  make,
  model,
  listing_country
FROM `{project_id}.{dataset_id}.vw_regression_dataset`
WHERE LOWER(make) = '{make}'
  AND LOWER(model) = '{model}'
  AND LOWER(listing_country) = '{country_name}'
  AND actual_price_eur IS NOT NULL AND actual_price_eur > 0
  AND mileage_km IS NOT NULL AND mileage_km > 0
  AND age_years IS NOT NULL AND age_years >= 0
  AND power_hp IS NOT NULL AND power_hp > 0
'''

df = run_query(sql_modeling)

if len(df) == 0:
    sql_modeling_no_country = f'''
    SELECT
      listing_id,
      actual_price_eur AS price_eur,
      mileage_km,
      age_years,
      power_hp,
      make,
      model,
      listing_country
    FROM `{project_id}.{dataset_id}.vw_regression_dataset`
    WHERE LOWER(make) = '{make}'
      AND LOWER(model) = '{model}'
      AND actual_price_eur IS NOT NULL AND actual_price_eur > 0
      AND mileage_km IS NOT NULL AND mileage_km > 0
      AND age_years IS NOT NULL AND age_years >= 0
      AND power_hp IS NOT NULL AND power_hp > 0
    '''
    df = run_query(sql_modeling_no_country)
    print('No rows found for configured country; fallback scope used make/model only.')

if len(df) == 0:
    raise ValueError('No rows available for configured regression scope. Check data and config values.')

print('Rows:', len(df))
print('Scope:', make, model, country_name)


## 3. Gráficos de dispersión rápidos (guardados en el disco)

Las parcelas se guardan para los materiales del curso; Los nombres de archivos deben permanecer sin cambios.

### Controles visuales
Los diagramas de dispersión muestran la relación entre el precio y cada característica. Estos gráficos se guardan para la enseñanza y no se utilizan en el modelo.

### ¿Por qué visualizar?
Los diagramas de dispersión ayudan a verificar que las relaciones sean razonables antes de modelar.

**Objetivo:** Visualizar las relaciones entre precio y características.
**Entradas:** df con precio, kilometraje, antigüedad y potencia.
**Proceso:** Traza gráficos de dispersión y guárdalos.
**Resultados:** Tres gráficos PNG para documentación.
**Por qué:** Las comprobaciones visuales ayudan a validar las relaciones entre funciones.


In [ ]:
# Objetivo: crear y guardar diagramas de dispersión para funciones clave.  #Objetivo
# Entrada: df con columnas de precio, kilometraje, antigüedad y potencia.  #Aporte
# Proceso: filas de muestra, trazar el precio frente a cada característica, guardar archivos PNG.  #Proceso
# Salida: Tres imágenes de diagramas de dispersión en el directorio de informes.  #Producción


def plot_feature_scatter(df_plot, x_col, x_label, reports_dir, filename):
    plt.figure(figsize=(8, 5))
    plt.scatter(df_plot[x_col], df_plot['price_eur'], alpha=0.4)
    plt.xlabel(x_label)
    plt.ylabel('Price (EUR)')
    plt.title(f'Price vs {x_label}')
    plt.tight_layout()
    plt.savefig(reports_dir / filename, dpi=150)
    plt.show()


PLOT_MAX_POINTS = 20000
plot_df = df.sample(n=min(PLOT_MAX_POINTS, len(df)), random_state=RANDOM_SEED)

plot_feature_scatter(plot_df, 'mileage_km', 'Mileage (km)', REPORTS_DIR,
                     f'scatter_price_vs_mileage_{TAG}.png')
plot_feature_scatter(plot_df, 'age_years', 'Age (years)', REPORTS_DIR,
                     f'scatter_price_vs_age_{TAG}.png')
plot_feature_scatter(plot_df, 'power_hp', 'Power (hp)', REPORTS_DIR,
                     f'scatter_price_vs_power_{TAG}.png')


## 4. Divisiones de prueba de tren

Mantenemos la selección de funciones explícita y visible para evitar confusión sobre las entradas.

### División de prueba de tren
Dividimos los datos en conjuntos de entrenamiento y prueba para poder evaluar el rendimiento del modelo en datos invisibles.

### ¿Por qué dividir?
Una división tren/prueba nos permite estimar qué tan bien se generaliza el modelo a datos invisibles.

**Objetivo:** Definir conjuntos de características y dividir datos para su evaluación.
**Entradas:** df con características numéricas y precio.
**Proceso:** Crear X/y y dividir en tren/prueba.
**Resultados:** Matrices de características y conjuntos de datos divididos.
**Por qué:** Las divisiones de entrenamiento/prueba proporcionan una evaluación imparcial.


In [ ]:
# Objetivo: construir matrices de características y una división de entrenamiento/prueba.  #Objetivo
# Entrada: df con características numéricas y precio objetivo.  #Aporte
# Proceso: cree subconjuntos de funciones y divide datos para realizar pruebas.  #Proceso
# Salida: X_* conjuntos de características y división de entrenamiento/prueba para el modelo multivariado.  #Producción

X_mileage = df[['mileage_km']]  # Conjunto de datos de característica única para kilometraje.
X_age = df[['age_years']]  # Conjunto de datos de una sola característica para la edad.
X_power = df[['power_hp']]  # Conjunto de datos de característica única para energía.
X_multi = df[['mileage_km', 'age_years', 'power_hp']]  # Conjunto de datos con múltiples funciones.
y = df['price_eur']  # Variable objetivo del precio.

Xb_train, Xb_test, y_train, y_test = train_test_split(
    X_multi, y, test_size=0.2, random_state=RANDOM_SEED
)  # Divida los datos multivariados en tren/prueba.


## 5. Ajustar regresiones simples (informe MAE + R2)

Sólo MAE y R2 se informan como métricas de regresión básicas para este curso.

### Entrenamiento modelo
Entrenamos tres modelos de una sola característica y un modelo de múltiples características, luego informamos MAE y R2 para cada uno.

### Lo que aprenden los modelos
Cada modelo univariado aprende una relación lineal con el precio. El modelo multivariado combina todas las características.

**Objetivo:** Entrenar y evaluar modelos de regresión lineal.
**Entradas:** Divisiones de entrenamiento/prueba para cada conjunto de funciones.
**Proceso:** Ajustar modelos y calcular MAE y R2.
**Resultados:** Métricas impresas para cada modelo.
**Por qué:** Las métricas muestran un rendimiento predictivo para comparar.


In [ ]:
# Objetivo: Entrenar modelos de regresión simples e informar MAE y R2.  #Objetivo
# Entrada: conjuntos de datos de características y objetivo y.  #Aporte
# Proceso: ajustar modelos, predecir divisiones de prueba, calcular métricas.  #Proceso
# Salida: Valores MAE y R2 impresos para cada modelo.  #Producción

# Modelo de solo kilometraje # Modelo que usa solo kilometraje.
Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    X_mileage, y, test_size=0.2, random_state=RANDOM_SEED
)  # Dividir datos para el modelo de kilometraje.
model_mileage = LinearRegression()  # Crear modelo de regresión lineal.
model_mileage.fit(Xm_train, ym_train)  # Ajustar el modelo a los datos de entrenamiento.

y_mileage_pred = model_mileage.predict(Xm_test)  # Predecir sobre los datos de prueba.
print('Mileage model:')  # Salida de etiquetas.
print('  R2 test:', r2_score(ym_test, y_mileage_pred))  # Imprimir métrica R2.
print('  MAE test (EUR):', mean_absolute_error(ym_test, y_mileage_pred))  # Imprimir MAE.

# Modelo solo por edad # Modelo que usa solo edad.
Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    X_age, y, test_size=0.2, random_state=RANDOM_SEED
)  # Dividir datos para el modelo de edad.
model_age = LinearRegression()  # Crear modelo de regresión lineal.
model_age.fit(Xa_train, ya_train)  # Ajustar el modelo a los datos de entrenamiento.

y_age_pred = model_age.predict(Xa_test)  # Predecir sobre los datos de prueba.
print('Age model:')  # Salida de etiquetas.
print('  R2 test:', r2_score(ya_test, y_age_pred))  # Imprimir métrica R2.
print('  MAE test (EUR):', mean_absolute_error(ya_test, y_age_pred))  # Imprimir MAE.

# Modelo de solo energía # Modelo que usa solo energía.
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    X_power, y, test_size=0.2, random_state=RANDOM_SEED
)  # Dividir datos para el modelo de potencia.
model_power = LinearRegression()  # Crear modelo de regresión lineal.
model_power.fit(Xp_train, yp_train)  # Ajustar el modelo a los datos de entrenamiento.

y_power_pred = model_power.predict(Xp_test)  # Predecir sobre los datos de prueba.
print('Power model:')  # Salida de etiquetas.
print('  R2 test:', r2_score(yp_test, y_power_pred))  # Imprimir métrica R2.
print('  MAE test (EUR):', mean_absolute_error(yp_test, y_power_pred))  # Imprimir MAE.

# Modelo multivariado # Modelo utilizando kilometraje, edad y potencia.
model_multi = LinearRegression()  # Crear regresión lineal multivariante.
model_multi.fit(Xb_train, y_train)  # Ajustar el modelo a los datos de entrenamiento.

y_multi_pred = model_multi.predict(Xb_test)  # Predecir sobre los datos de prueba.
print('Multivariate model:')  # Salida de etiquetas.
print('  R2 test:', r2_score(y_test, y_multi_pred))  # Imprimir métrica R2.
print('  MAE test (EUR):', mean_absolute_error(y_test, y_multi_pred))  # Imprimir MAE.


## 6. Compare modelos en la misma división de prueba (ligero)

La tabla comparativa facilita ver qué modelo funciona mejor.

### Tabla comparativa
Construimos un pequeño DataFrame para comparar los modelos en las mismas divisiones de prueba.

### Tabla de salida
La tabla de comparación resume el rendimiento del modelo para que los estudiantes puedan elegir el modelo más sólido.

**Objetivo:** Resumir el rendimiento del modelo en una tabla.
**Entradas:** Métricas de cada modelo.
**Proceso:** Reúna métricas en un DataFrame.
**Salidas:** Tabla comparativa impresa.
**Por qué:** Una tabla aclara la comparación de modelos para los estudiantes.


In [ ]:
# Objetivo: comparar el rendimiento del modelo en una pequeña tabla resumen.  #Objetivo
# Entrada: Predicciones y objetivos de cada modelo.  #Aporte
# Proceso: Calcule R2 y MAE para cada modelo y ensamble un DataFrame.  #Proceso
# Salida: metrics_df impreso en la pantalla.  #Producción

metrics = []  # Inicializar lista de diccionarios de métricas.

metrics.append({
    'model': 'price ~ mileage',
    'r2_test': r2_score(ym_test, y_mileage_pred),
    'mae_test': mean_absolute_error(ym_test, y_mileage_pred),
})  # Almacenar métricas para el modelo de kilometraje.

metrics.append({
    'model': 'price ~ age',
    'r2_test': r2_score(ya_test, y_age_pred),
    'mae_test': mean_absolute_error(ya_test, y_age_pred),
})  # Almacenar métricas para el modelo de edad.

metrics.append({
    'model': 'price ~ power',
    'r2_test': r2_score(yp_test, y_power_pred),
    'mae_test': mean_absolute_error(yp_test, y_power_pred),
})  # Almacenar métricas para el modelo de energía.

metrics.append({
    'model': 'price ~ mileage + age + power',
    'r2_test': r2_score(y_test, y_multi_pred),
    'mae_test': mean_absolute_error(y_test, y_multi_pred),
})  # Almacenar métricas para el modelo multivariado.

metrics_df = pd.DataFrame(metrics)  # Convierta la lista de métricas en un DataFrame.
print(metrics_df)  # Mostrar la tabla de comparación.


## 7. Entrene el modelo final y escriba predicciones en BigQuery

Las predicciones se conservan en una tabla fija BigQuery que se utiliza en sentido descendente.

### Persistencia
Ajustamos un modelo final a todos los datos y guardamos las predicciones por listado en BigQuery.

### Salida final
La tabla de predicción se guarda en BigQuery y se puede utilizar para paneles o análisis posteriores.

**Objetivo:** Ajustar un modelo final y guardar las predicciones en BigQuery.
**Entradas:** Características completas del conjunto de datos y objetivo.
**Proceso:** Entrene en todas las filas y escriba la tabla de predicciones.
**Salidas:** BigQuery tabla `fact_expected_price_predictions`.
**Por qué:** Las predicciones persistentes se pueden utilizar en análisis posteriores.


In [ ]:
# Objetivo: entrenar un modelo final con todos los datos y escribir las salidas expect-price en BigQuery.  #Objetivo
# Entrada: conjunto completo de funciones X_multi y objetivo y.  #Aporte
# Proceso: ajustar el modelo, generar precios esperados, crear una tabla de resultados, escribir en BigQuery.  #Proceso
# Salida: tabla BigQuery fact_expected_price_predictions reemplazada con nuevas predicciones.  #Producción

model_full = LinearRegression()
model_full.fit(X_multi, y)

expected_price = model_full.predict(X_multi)

predictions_df = df[['listing_id']].copy()
predictions_df['expected_price_eur'] = expected_price.astype(float)

pred_table = f"{project_id}.{dataset_id}.fact_expected_price_predictions"
job_config = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')
client.load_table_from_dataframe(predictions_df, pred_table, job_config=job_config).result()
print('Predictions table replaced:', pred_table)
